# Canonical AME(4,3) Bell baseline

This notebook prepares a deterministic canonical direct-basis input for the AME(4,3) Bell experiment. By default, Run All submits only the local Aer baseline; IQM and PiastQ submit remote jobs only after `RUN_IQM` or `RUN_PIASTQ` is set to `True`. Credentials remain provider/environment-only and are never embedded or persisted by this notebook.

In [ ]:
import hashlib
import json
import stat
import sys
from pathlib import Path
from uuid import uuid4

import numpy as np
import qiskit.qpy as qpy
from qiskit.quantum_info import Statevector


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "qudits_on_qubits").is_dir():
            return candidate
    raise RuntimeError(
        "Cannot find repository root. Start from this repository or a descendant "
        "containing pyproject.toml and src/qudits_on_qubits."
    )

def _checked_iqm_path(path, expected, description):
    path = Path(path)
    try:
        metadata = path.lstat()
    except FileNotFoundError:
        return None
    except OSError as error:
        raise RuntimeError(f'Cannot inspect {description}: {path}') from error
    reparse_attribute = getattr(stat, 'FILE_ATTRIBUTE_REPARSE_POINT', 0x400)
    if stat.S_ISLNK(metadata.st_mode) or bool(getattr(metadata, 'st_file_attributes', 0) & reparse_attribute):
        raise RuntimeError(f'Refusing symlink or reparse {description}: {path}')
    if expected == 'file' and not stat.S_ISREG(metadata.st_mode):
        raise RuntimeError(f'Malformed Git metadata or IQM .env candidate: expected file at {path}')
    if expected == 'dir' and not stat.S_ISDIR(metadata.st_mode):
        raise RuntimeError(f'Malformed Git metadata: expected directory at {path}')
    return path.resolve()


def resolve_iqm_env_path(repo_root):
    repo_root = Path(repo_root)
    local_env = _checked_iqm_path(repo_root / '.env', 'file', 'IQM .env candidate')
    if local_env is not None:
        return local_env
    git_metadata = repo_root / '.git'
    metadata = _checked_iqm_path(git_metadata, 'any', 'Git metadata')
    if metadata is None:
        raise RuntimeError(f'Missing Git metadata; cannot resolve IQM .env for {repo_root}')
    owning_repo = repo_root.resolve()
    if stat.S_ISREG(git_metadata.lstat().st_mode):
        try:
            gitdir_line = metadata.read_text(encoding='utf-8').strip()
        except OSError as error:
            raise RuntimeError(f'Malformed Git metadata: cannot read {git_metadata}') from error
        if not gitdir_line.lower().startswith('gitdir:'):
            raise RuntimeError(f'Malformed Git metadata: expected gitdir in {git_metadata}')
        gitdir_value = gitdir_line[7:].strip()
        if not gitdir_value:
            raise RuntimeError(f'Malformed Git metadata: empty gitdir in {git_metadata}')
        gitdir = Path(gitdir_value)
        if not gitdir.is_absolute():
            gitdir = metadata.parent / gitdir
        gitdir = _checked_iqm_path(gitdir, 'dir', 'Git worktree metadata')
        if gitdir is None:
            raise RuntimeError(f'Malformed Git metadata: missing worktree directory for {git_metadata}')
        commondir = _checked_iqm_path(gitdir / 'commondir', 'file', 'Git commondir metadata')
        if commondir is None:
            raise RuntimeError(f'Malformed Git metadata: missing commondir in {gitdir}')
        try:
            commondir_value = commondir.read_text(encoding='utf-8').strip()
        except OSError as error:
            raise RuntimeError(f'Malformed Git metadata: cannot read {commondir}') from error
        if not commondir_value:
            raise RuntimeError(f'Malformed Git metadata: empty commondir in {commondir}')
        common_git_dir = Path(commondir_value)
        if not common_git_dir.is_absolute():
            common_git_dir = gitdir / common_git_dir
        common_git_dir = _checked_iqm_path(common_git_dir, 'dir', 'Git common directory')
        if common_git_dir is None:
            raise RuntimeError(f'Malformed Git metadata: missing common directory for {gitdir}')
        if common_git_dir.name != '.git':
            raise RuntimeError('Cannot use non-.git common directory for IQM .env fallback; provide checkout-local .env or explicit env_path.')
        owner_git_dir = _checked_iqm_path(common_git_dir.parent / '.git', 'dir', 'Git common directory')
        if owner_git_dir != common_git_dir:
            raise RuntimeError('Cannot validate non-bare owning repository for IQM .env fallback; provide checkout-local .env or explicit env_path.')
        owning_repo = _checked_iqm_path(common_git_dir.parent, 'dir', 'owning repository')
        if owning_repo is None:
            raise RuntimeError('Cannot validate owning repository for IQM .env fallback; provide checkout-local .env or explicit env_path.')
    candidates = [repo_root / '.env']
    if owning_repo != repo_root.resolve():
        candidates.append(owning_repo / '.env')
    for candidate in candidates:
        checked = _checked_iqm_path(candidate, 'file', 'IQM .env candidate')
        if checked is not None:
            return checked
    raise RuntimeError(f'Cannot find IQM .env file for checkout or owning repository: {repo_root}')


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qudits_on_qubits.reference_experiments import get_encoding, get_reference_experiment
from qudits_on_qubits.benchmarks.direct_basis.circuits import build_direct_basis_graph_state_circuit
from qudits_on_qubits.experiments import (
    AerIdeal,
    BootstrapConfig,
    ExperimentSpec,
    IQMHardware,
    MitigationConfig,
    PathBasis,
    PiastQHardware,
    run_experiment,
)

In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


class CanonicalBasisFormatError(RuntimeError):
    pass


def is_symlink_or_reparse(path):
    try:
        metadata = Path(path).lstat()
    except OSError as error:
        raise RuntimeError(f"unable to inspect canonical basis path: {path}") from error
    reparse_attribute = getattr(stat, "FILE_ATTRIBUTE_REPARSE_POINT", 0x400)
    file_attributes = getattr(metadata, "st_file_attributes", 0)
    return stat.S_ISLNK(metadata.st_mode) or bool(file_attributes & reparse_attribute)


def assert_safe_path_components(repo_root, path):
    repo_root = Path(repo_root)
    path = Path(path)
    try:
        relative_parts = path.relative_to(repo_root).parts
        resolved_root = repo_root.resolve(strict=True)
    except (OSError, ValueError) as error:
        raise RuntimeError(f'canonical basis path is outside an available repository root: {path}') from error
    current = repo_root
    for part in ('', *relative_parts):
        if part:
            current = current / part
        try:
            current.lstat()
        except FileNotFoundError:
            continue
        except OSError as error:
            raise RuntimeError(f'unable to inspect canonical basis path: {current}') from error
        if is_symlink_or_reparse(current):
            raise RuntimeError('canonical basis path must not contain a symlink or reparse point')
        try:
            resolved_current = current.resolve(strict=True)
        except OSError as error:
            raise RuntimeError(f'unable to resolve canonical basis path: {current}') from error
        if not resolved_current.is_relative_to(resolved_root):
            raise RuntimeError('canonical basis path escapes the repository root')


def ensure_safe_directory(repo_root, directory):
    repo_root = Path(repo_root)
    directory = Path(directory)
    assert_safe_path_components(repo_root, directory)
    current = repo_root
    for part in directory.relative_to(repo_root).parts:
        current = current / part
        if not current.exists():
            try:
                current.mkdir()
            except FileExistsError:
                pass
        assert_safe_path_components(repo_root, current)
        if not current.is_dir():
            raise RuntimeError(f'canonical basis path component must be a directory: {current}')


def load_single_circuit(path):
    try:
        with Path(path).open("rb") as handle:
            circuits = qpy.load(handle)
    except Exception as error:
        raise RuntimeError(f"unable to load canonical basis QPY: {path}") from error
    if len(circuits) != 1:
        raise RuntimeError(f"canonical basis QPY must contain exactly one circuit, found {len(circuits)}")
    return circuits[0]


def validate_canonical_basis(repo_root, directory, expected_encoding, expected_circuit):
    directory = Path(directory)
    assert_safe_path_components(repo_root, directory)
    required_files = {"graph_state_direct_basis.qpy", "E.npy", "metadata.json"}
    try:
        actual_files = {path.name for path in directory.iterdir()}
    except OSError as error:
        raise RuntimeError(f"canonical basis directory is unavailable: {directory}") from error
    if actual_files != required_files:
        raise RuntimeError(
            "canonical basis files must be exactly graph_state_direct_basis.qpy, E.npy, "
            f"and metadata.json; found {sorted(actual_files)}"
        )
    for required_file in required_files:
        assert_safe_path_components(repo_root, directory / required_file)

    encoding_path = directory / "E.npy"
    try:
        encoding = np.load(encoding_path, allow_pickle=False)
    except Exception as error:
        raise RuntimeError(f"canonical basis encoding is invalid: {encoding_path}") from error
    if encoding.shape != (4, 3):
        raise RuntimeError(f"canonical basis encoding must have shape (4, 3), got {encoding.shape}")
    try:
        is_finite = np.isfinite(encoding).all()
    except TypeError as error:
        raise RuntimeError("canonical basis encoding must be numeric and finite") from error
    if not is_finite:
        raise RuntimeError("canonical basis encoding must be finite")
    if not np.allclose(encoding.conj().T @ encoding, np.eye(3), atol=1e-12, rtol=0):
        raise RuntimeError("canonical basis encoding must be an isometry")
    if not np.array_equal(encoding, expected_encoding):
        raise RuntimeError("canonical basis encoding does not match canonical_ez")

    circuit_path = directory / "graph_state_direct_basis.qpy"
    circuit = load_single_circuit(circuit_path)
    if circuit.num_qubits != 8 or circuit.num_clbits != 0:
        raise RuntimeError("canonical basis QPY must contain one unmeasured eight-qubit circuit")
    for instruction in circuit.data:
        operation = instruction.operation
        if operation.name in {"measure", "reset"}:
            raise RuntimeError("canonical basis QPY must not contain measurements or resets")
        if getattr(operation, "condition", None) is not None:
            raise RuntimeError("canonical basis QPY must not contain conditioned instructions")
        if getattr(operation, "blocks", ("")):
            raise RuntimeError("canonical basis QPY must not contain control flow")
    try:
        same_state = Statevector.from_instruction(circuit).equiv(
            Statevector.from_instruction(expected_circuit)
        )
    except Exception as error:
        raise RuntimeError("canonical basis QPY circuit cannot be validated as a state preparation") from error
    if not same_state:
        raise RuntimeError("canonical basis QPY circuit does not match the canonical graph state")

    metadata_path = directory / "metadata.json"
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    except Exception as error:
        raise RuntimeError(f"canonical basis metadata is invalid: {metadata_path}") from error
    expected_metadata = {
        "schema": "qoq-reference-basis-v1",
        "state": "ame43",
        "encoding_id": "canonical_ez",
        "num_qubits": 8,
        "encoding_shape": [4, 3],
        "files": {
            "graph_state_direct_basis.qpy": {"sha256": sha256_file(circuit_path)},
            "E.npy": {"sha256": sha256_file(encoding_path)},
        },
    }
    if metadata != expected_metadata:
        raise RuntimeError("canonical basis metadata does not match the validated bundle")
    actual_operation_types = tuple(type(item.operation) for item in circuit.data)
    expected_operation_types = tuple(type(item.operation) for item in expected_circuit.data)
    if actual_operation_types != expected_operation_types:
        raise CanonicalBasisFormatError("canonical basis QPY gate format is stale")


def prepare_canonical_basis(repo_root):
    repo_root = Path(repo_root)
    expected_encoding = get_encoding("canonical_ez").as_array()
    expected_circuit = build_direct_basis_graph_state_circuit("ame43", expected_encoding)
    parent = repo_root / "experiment_inputs" / "reference_bases" / "ame43"
    directory = parent / "canonical_ez"
    ensure_safe_directory(repo_root, parent)
    assert_safe_path_components(repo_root, directory)

    rebuild_legacy = False
    if directory.exists():
        if is_symlink_or_reparse(directory):
            raise RuntimeError("canonical basis directory must not be a symlink or reparse point")
        try:
            validate_canonical_basis(repo_root, directory, expected_encoding, expected_circuit)
        except CanonicalBasisFormatError:
            rebuild_legacy = True
        else:
            return directory

    staging_directory = parent / f".canonical_ez.tmp-{uuid4().hex}"
    assert_safe_path_components(repo_root, staging_directory)
    staging_directory.mkdir()
    assert_safe_path_components(repo_root, staging_directory)
    staging_files = (
        staging_directory / "graph_state_direct_basis.qpy",
        staging_directory / "E.npy",
        staging_directory / "metadata.json",
    )

    def cleanup_staging():
        assert_safe_path_components(repo_root, staging_directory)
        for staging_file in staging_files:
            if staging_file.exists():
                assert_safe_path_components(repo_root, staging_file)
                staging_file.unlink()
        if staging_directory.exists():
            assert_safe_path_components(repo_root, staging_directory)
            staging_directory.rmdir()

    try:
        qpy_path, encoding_path, metadata_path = staging_files
        for staging_file in staging_files:
            assert_safe_path_components(repo_root, staging_file)
        with qpy_path.open("wb") as handle:
            qpy.dump(expected_circuit, handle)
        with encoding_path.open("wb") as handle:
            np.save(handle, expected_encoding, allow_pickle=False)
        metadata = {
            "schema": "qoq-reference-basis-v1",
            "state": "ame43",
            "encoding_id": "canonical_ez",
            "num_qubits": 8,
            "encoding_shape": [4, 3],
            "files": {
                "graph_state_direct_basis.qpy": {"sha256": sha256_file(qpy_path)},
                "E.npy": {"sha256": sha256_file(encoding_path)},
            },
        }
        metadata_path.write_text(
            json.dumps(metadata, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )

        validate_canonical_basis(repo_root, staging_directory, expected_encoding, expected_circuit)
        backup_directory = None
        if rebuild_legacy:
            candidate = parent / f".canonical_ez.legacy-{uuid4().hex}"
            assert_safe_path_components(repo_root, directory)
            assert_safe_path_components(repo_root, candidate)
            try:
                directory.rename(candidate)
            except FileNotFoundError:
                pass
            else:
                assert_safe_path_components(repo_root, candidate)
                backup_directory = candidate
        try:
            assert_safe_path_components(repo_root, staging_directory)
            assert_safe_path_components(repo_root, directory)
            staging_directory.rename(directory)
        except FileExistsError:
            cleanup_staging()
            validate_canonical_basis(repo_root, directory, expected_encoding, expected_circuit)
        except BaseException:
            if backup_directory is not None and not directory.exists():
                assert_safe_path_components(repo_root, backup_directory)
                assert_safe_path_components(repo_root, directory)
                backup_directory.rename(directory)
            raise
        if backup_directory is not None:
            assert_safe_path_components(repo_root, backup_directory)
            for backup_file in backup_directory.iterdir():
                assert_safe_path_components(repo_root, backup_file)
                backup_file.unlink()
            assert_safe_path_components(repo_root, backup_directory)
            backup_directory.rmdir()
        return directory
    finally:
        cleanup_staging()

In [ ]:
CANONICAL_BASIS_DIRECTORY = prepare_canonical_basis(REPO_ROOT)
CANONICAL_BASIS_DIRECTORY

## Shared configuration

The canonical reference, uncertainty settings, and hardware mitigation policy are shared across the three backend baselines.

In [ ]:
SHOTS = 100
UNCERTAINTY = BootstrapConfig(samples=2_000, seed=7)
HARDWARE_MITIGATION = MitigationConfig(readout=True, zne=True, zne_factors=(1, 3, 5))
REFERENCE = get_reference_experiment('ame43')
RESULTS = {}

## Aer ideal baseline

This unguarded local baseline records the canonical ideal-backend result.

In [ ]:
AER_SPEC = ExperimentSpec(
    state='ame43',
    basis=PathBasis(CANONICAL_BASIS_DIRECTORY),
    backend=AerIdeal(seed_simulator=11),
    shots=SHOTS,
    uncertainty=UNCERTAINTY,
    tags={'baseline': 'canonical_ez', 'backend': 'aer_ideal'},
)
AER_RESULT = run_experiment(AER_SPEC, repo_root=REPO_ROOT)
RESULTS['aer_ideal'] = AER_RESULT

## IQM Garnet baseline

Submission is opt-in; the default keeps this hardware run skipped.

In [ ]:
RUN_IQM = False

if RUN_IQM:
    IQM_ENV_PATH = resolve_iqm_env_path(REPO_ROOT)
    IQM_SPEC = ExperimentSpec(
        state='ame43',
        basis=PathBasis(CANONICAL_BASIS_DIRECTORY),
        backend=IQMHardware(device='garnet', use_metrics=True, env_path=IQM_ENV_PATH),
        shots=SHOTS,
        mitigation=HARDWARE_MITIGATION,
        uncertainty=UNCERTAINTY,
        tags={'baseline': 'canonical_ez', 'backend': 'iqm_garnet'},
    )
    IQM_RESULT = run_experiment(IQM_SPEC, repo_root=REPO_ROOT)
    RESULTS['iqm_garnet'] = IQM_RESULT
else:
    print('IQM Garnet skipped; set RUN_IQM = True to submit.')

## PiastQ baseline

Submission is opt-in; the default keeps this hardware run skipped.

In [ ]:
RUN_PIASTQ = False

if RUN_PIASTQ:
    PIASTQ_SPEC = ExperimentSpec(
        state='ame43',
        basis=PathBasis(CANONICAL_BASIS_DIRECTORY),
        backend=PiastQHardware(mode='managed', owner='notebook'),
        shots=SHOTS,
        mitigation=HARDWARE_MITIGATION,
        uncertainty=UNCERTAINTY,
        tags={'baseline': 'canonical_ez', 'backend': 'piastq'},
    )
    PIASTQ_RESULT = run_experiment(PIASTQ_SPEC, repo_root=REPO_ROOT)
    RESULTS['piastq'] = PIASTQ_RESULT
else:
    print('PiastQ skipped; set RUN_PIASTQ = True to submit.')

## Comparison and safety

The summary reports every planned backend. Hardware remains explicit when intentionally skipped, and it reuses runner-produced estimates and mitigation outputs without recomputation.

In [ ]:
def summarize_results(results, reference):
    rows = []
    for backend, missing_status in (
        ('aer_ideal', 'not_run'),
        ('iqm_garnet', 'skipped'),
        ('piastq', 'skipped'),
    ):
        result = results.get(backend)
        row = {
            'backend': backend,
            'status': missing_status if result is None else result.status.value,
            'raw': None if result is None else result.values.get('raw'),
            'readout_mitigated': None if result is None else result.values.get('readout_mitigated'),
            'zne': None if result is None else result.values.get('zne'),
            'zne_readout_mitigated': None if result is None else result.values.get('zne_readout_mitigated'),
            'diagnostics': None if result is None else result.values.get('diagnostics'),
            'leakage_rate': None if result is None else result.values.get('leakage_rate'),
            'classical_bound': reference.bell_functional.classical_bound,
            'ideal_bell_value': reference.expected.ideal_bell_value,
            'artifact_dir': None if result is None else str(result.artifact_dir),
        }
        rows.append(row)
    return rows

In [ ]:
SUMMARY = summarize_results(RESULTS, REFERENCE)
SUMMARY